#첫번째는 affine 기반 회전을 해보았으나 실패

In [3]:
# -*- coding: utf-8 -*-
"""
양상추 정면 베드 4점 기반 affine/약한 정렬 + QC 엑셀 출력

핵심 목적
1) perspective처럼 강하게 직사각형으로 펴지지 않음
2) 4점을 이용해 날짜별 기울기/높이 흔들림만 줄이는 '약한 정렬'
3) 전/후 비교 이미지 저장
4) QC 지표(before/after) 엑셀 저장

왜 affine/약한 정렬인가?
- perspective warp는 사다리꼴을 강제로 직사각형으로 바꾸면서 모양 왜곡이 커질 수 있음
- 지금 목표는 '정렬 안정화'이지 '완전 정사영 복원'이 아님
- 그래서 이 코드는
  (a) 상단선이 너무 들쭉날쭉한 문제
  (b) 날짜별 좌우 기울기 변화
  (c) 세로 위치 흔들림
  만 줄이는 방향으로 동작함

정렬 방식
- p1, p2, p3, p4 = 좌상, 우상, 우하, 좌하
- 상단 중점(top_mid), 하단 중점(bot_mid)을 구함
- 이 두 점을 잇는 중심축이 '세로 방향'이 되도록 회전
- 추가로 상단선/하단선의 평균 기울기를 구해 과한 비틀림이 없는지 QC로 기록
- 이미지는 회전만 수행하므로 모양 찌그러짐이 매우 적음

입력 CSV 필수 여부
- 이 단계에서는 '이미지별 4점 좌표'가 들어있는 CSV가 사실상 필요함
- 예: image_name, image_path, p1_x, p1_y, ..., p4_x, p4_y
- 네가 올린 per_image_results_v3.csv 구조면 바로 사용 가능

권장 실행 순서
1) 샘플 30장만 돌려서 before/after 확인
2) QC 엑셀 열어서 slope, bbox, 잘림 정도 확인
3) 괜찮으면 전체 적용

주의
- 이 코드는 '약한 정렬'이라 crop을 다시 강하게 자르지 않음
- 그래서 상단/하단 잘림을 악화시키는 위험이 perspective보다 작음
- 다만 회전 후 빈 여백이 생길 수 있어 border_mode를 reflect로 둠
"""

import os
import math
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# =========================
# 0. 경로/설정
# =========================
CSV_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_final/per_image_results_v3.csv"   # 4점 CSV
IMAGE_ROOT = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/260402"                           # image_path가 절대경로면 무시됨
OUT_ROOT = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine_v1"

# 우선 샘플 QC만 볼 때
RUN_SAMPLE_ONLY = True
SAMPLE_N = 30
RANDOM_SEED = 42

# 병렬 처리
N_WORKERS = 8

# 비교 이미지 저장 개수
SAVE_COMPARE_MAX = 30

# 회전 후 바깥 테두리 채우기
BORDER_MODE = cv2.BORDER_REFLECT

# 너무 이상한 샘플 걸러내는 기준(필요시 조정)
MAX_ABS_TOP_SLOPE = 25
MAX_ABS_BOTTOM_SLOPE = 25
MIN_QUAD_AREA = 5000


In [4]:
# =========================
# 1. 유틸
# =========================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def resolve_image_path(p):
    p = str(p)
    if os.path.exists(p):
        return p
    p2 = os.path.join(IMAGE_ROOT, p.lstrip("/"))
    return p2


def order_points_if_needed(pts):
    """
    pts shape = (4,2)
    가능한 한 [좌상, 우상, 우하, 좌하] 순서로 정렬
    이미 그 순서면 거의 유지됨.
    """
    pts = np.array(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)

    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]

    return np.array([tl, tr, br, bl], dtype=np.float32)


def line_angle_deg(p_a, p_b):
    dx = float(p_b[0] - p_a[0])
    dy = float(p_b[1] - p_a[1])
    return math.degrees(math.atan2(dy, dx))


def rotate_points(pts, M):
    pts_h = np.hstack([pts.astype(np.float32), np.ones((len(pts), 1), dtype=np.float32)])
    out = (M @ pts_h.T).T
    return out


def make_compare_canvas(img_before, img_after, pts_before, pts_after, text_lines=None):
    before = img_before.copy()
    after = img_after.copy()

    pts_before = pts_before.astype(int)
    pts_after = pts_after.astype(int)

    cv2.polylines(before, [pts_before], True, (0, 255, 255), 2)
    cv2.polylines(after, [pts_after], True, (0, 255, 255), 2)

    for i, (x, y) in enumerate(pts_before, start=1):
        cv2.circle(before, (x, y), 5, (0, 0, 255), -1)
        cv2.putText(before, f"p{i}", (x+5, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    for i, (x, y) in enumerate(pts_after, start=1):
        cv2.circle(after, (x, y), 5, (0, 0, 255), -1)
        cv2.putText(after, f"p{i}", (x+5, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    h = max(before.shape[0], after.shape[0])
    w1 = before.shape[1]
    w2 = after.shape[1]

    if before.shape[0] != h:
        tmp = np.zeros((h, w1, 3), dtype=np.uint8)
        tmp[:before.shape[0], :before.shape[1]] = before
        before = tmp
    if after.shape[0] != h:
        tmp = np.zeros((h, w2, 3), dtype=np.uint8)
        tmp[:after.shape[0], :after.shape[1]] = after
        after = tmp

    canvas = np.concatenate([before, after], axis=1)

    cv2.putText(canvas, "BEFORE", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
    cv2.putText(canvas, "AFTER", (w1 + 20, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)

    if text_lines:
        y0 = 70
        for line in text_lines:
            cv2.putText(canvas, str(line), (20, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            y0 += 28

    return canvas



In [5]:
# =========================
# 2. affine/약한 정렬 핵심
# =========================
def weak_affine_align(img, pts):
    """
    핵심 아이디어:
    - top_mid -> bot_mid 중심축이 세로 방향이 되도록 회전만 수행
    - perspective처럼 강한 변형 없음
    - 결과적으로 날짜별 흔들림 감소 + 구조 왜곡 최소화
    """
    pts = order_points_if_needed(pts)

    top_mid = (pts[0] + pts[1]) / 2.0
    bot_mid = (pts[3] + pts[2]) / 2.0

    axis_vec = bot_mid - top_mid
    axis_angle_deg = math.degrees(math.atan2(axis_vec[1], axis_vec[0]))

    # 세로축(90도)에 맞추기 위한 회전량
    rot_deg = 90.0 - axis_angle_deg

    h, w = img.shape[:2]
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, rot_deg, 1.0)

    aligned = cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=BORDER_MODE,
    )

    pts_after = rotate_points(pts, M)
    return aligned, pts, pts_after, M, rot_deg


# =========================
# 3. QC 계산
# =========================
def compute_qc_metrics(img, pts):
    pts = np.array(pts, dtype=np.float32)
    h, w = img.shape[:2]

    top_slope = line_angle_deg(pts[0], pts[1])
    bottom_slope = line_angle_deg(pts[3], pts[2])
    left_height = float(np.linalg.norm(pts[3] - pts[0]))
    right_height = float(np.linalg.norm(pts[2] - pts[1]))
    top_width = float(np.linalg.norm(pts[1] - pts[0]))
    bottom_width = float(np.linalg.norm(pts[2] - pts[3]))
    centroid = pts.mean(axis=0)
    area = float(cv2.contourArea(pts.astype(np.float32)))

    x_min = float(np.min(pts[:, 0]))
    x_max = float(np.max(pts[:, 0]))
    y_min = float(np.min(pts[:, 1]))
    y_max = float(np.max(pts[:, 1]))

    margin_left = x_min
    margin_right = w - x_max
    margin_top = y_min
    margin_bottom = h - y_max

    # 화면 경계에 너무 붙었는지
    border_touch = int(
        (margin_left < 5) or (margin_right < 5) or (margin_top < 5) or (margin_bottom < 5)
    )

    return {
        "top_slope_deg": top_slope,
        "bottom_slope_deg": bottom_slope,
        "left_height_px": left_height,
        "right_height_px": right_height,
        "top_width_px": top_width,
        "bottom_width_px": bottom_width,
        "height_ratio_lr": left_height / (right_height + 1e-6),
        "width_ratio_tb": top_width / (bottom_width + 1e-6),
        "centroid_x": float(centroid[0]),
        "centroid_y": float(centroid[1]),
        "quad_area": area,
        "bbox_xmin": x_min,
        "bbox_xmax": x_max,
        "bbox_ymin": y_min,
        "bbox_ymax": y_max,
        "margin_left": margin_left,
        "margin_right": margin_right,
        "margin_top": margin_top,
        "margin_bottom": margin_bottom,
        "border_touch": border_touch,
        "img_w": w,
        "img_h": h,
    }


def make_qc_flag(before_m, after_m):
    flags = []

    # 정렬 개선 여부: 상단/하단 기울기 절댓값 감소
    before_slope_score = abs(before_m["top_slope_deg"]) + abs(before_m["bottom_slope_deg"])
    after_slope_score = abs(after_m["top_slope_deg"]) + abs(after_m["bottom_slope_deg"])
    slope_improved = after_slope_score < before_slope_score

    if not slope_improved:
        flags.append("slope_not_improved")

    # 과도한 기울기 남아 있으면 표시
    if abs(after_m["top_slope_deg"]) > MAX_ABS_TOP_SLOPE:
        flags.append("top_slope_still_large")
    if abs(after_m["bottom_slope_deg"]) > MAX_ABS_BOTTOM_SLOPE:
        flags.append("bottom_slope_still_large")

    # 면적이 너무 작으면 4점 자체 문제 가능성
    if before_m["quad_area"] < MIN_QUAD_AREA:
        flags.append("quad_area_too_small")

    # 경계에 닿으면 회전 후 더 잘릴 위험
    if after_m["border_touch"] == 1:
        flags.append("border_touch_after")

    if len(flags) == 0:
        return "ok"
    return "|".join(flags)


# =========================
# 4. 1장 처리
# =========================
def process_one(row, save_compare=False, compare_dir=None, aligned_dir=None):
    image_name = row.get("image_name", "unknown")
    image_path = resolve_image_path(row.get("image_path", ""))

    if not os.path.exists(image_path):
        return {
            "image_name": image_name,
            "image_path": image_path,
            "status": "image_not_found",
        }

    img = cv2.imread(image_path)
    if img is None:
        return {
            "image_name": image_name,
            "image_path": image_path,
            "status": "imread_failed",
        }

    pts = np.array([
        [row["p1_x"], row["p1_y"]],
        [row["p2_x"], row["p2_y"]],
        [row["p3_x"], row["p3_y"]],
        [row["p4_x"], row["p4_y"]],
    ], dtype=np.float32)

    aligned, pts_before, pts_after, M, rot_deg = weak_affine_align(img, pts)

    before_m = compute_qc_metrics(img, pts_before)
    after_m = compute_qc_metrics(aligned, pts_after)
    qc_flag = make_qc_flag(before_m, after_m)

    if aligned_dir is not None:
        out_img_path = os.path.join(aligned_dir, image_name)
        cv2.imwrite(out_img_path, aligned)
    else:
        out_img_path = ""

    if save_compare and compare_dir is not None:
        text_lines = [
            f"rot_deg={rot_deg:.2f}",
            f"top_slope: {before_m['top_slope_deg']:.2f} -> {after_m['top_slope_deg']:.2f}",
            f"bottom_slope: {before_m['bottom_slope_deg']:.2f} -> {after_m['bottom_slope_deg']:.2f}",
            f"qc_flag={qc_flag}",
        ]
        canvas = make_compare_canvas(img, aligned, pts_before, pts_after, text_lines=text_lines)
        cv2.imwrite(os.path.join(compare_dir, image_name), canvas)

    result = {
        "image_name": image_name,
        "image_path": image_path,
        "status": "ok",
        "aligned_path": out_img_path,
        "rot_deg": rot_deg,
        "qc_flag": qc_flag,
    }

    for k, v in before_m.items():
        result[f"before_{k}"] = v
    for k, v in after_m.items():
        result[f"after_{k}"] = v

    # 좌표도 저장
    for i, (x, y) in enumerate(pts_before, start=1):
        result[f"before_p{i}_x"] = float(x)
        result[f"before_p{i}_y"] = float(y)
    for i, (x, y) in enumerate(pts_after, start=1):
        result[f"after_p{i}_x"] = float(x)
        result[f"after_p{i}_y"] = float(y)

    return result




In [6]:
# =========================
# 5. 메인 실행
# =========================
def main():
    ensure_dir(OUT_ROOT)
    compare_dir = os.path.join(OUT_ROOT, "compare")
    aligned_dir = os.path.join(OUT_ROOT, "aligned")
    ensure_dir(compare_dir)
    ensure_dir(aligned_dir)

    df = pd.read_csv(CSV_PATH)

    required_cols = [
        "image_name", "image_path",
        "p1_x", "p1_y", "p2_x", "p2_y", "p3_x", "p3_y", "p4_x", "p4_y"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"CSV에 필요한 컬럼이 없음: {missing}")

    # 샘플 실행
    if RUN_SAMPLE_ONLY:
        random.seed(RANDOM_SEED)
        if len(df) > SAMPLE_N:
            df_run = df.sample(SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
        else:
            df_run = df.copy().reset_index(drop=True)
    else:
        df_run = df.copy().reset_index(drop=True)

    results = []
    save_compare_names = set(df_run["image_name"].head(SAVE_COMPARE_MAX).tolist())

    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = []
        for _, row in df_run.iterrows():
            save_compare = row["image_name"] in save_compare_names
            futures.append(
                ex.submit(
                    process_one,
                    row,
                    save_compare,
                    compare_dir,
                    aligned_dir,
                )
            )

        for fut in tqdm(as_completed(futures), total=len(futures), desc="affine qc"):
            results.append(fut.result())

    out_df = pd.DataFrame(results)

    # 개선 정도 수치화
    if "before_top_slope_deg" in out_df.columns:
        out_df["top_slope_abs_improve"] = out_df["before_top_slope_deg"].abs() - out_df["after_top_slope_deg"].abs()
        out_df["bottom_slope_abs_improve"] = out_df["before_bottom_slope_deg"].abs() - out_df["after_bottom_slope_deg"].abs()
        out_df["slope_total_improve"] = (
            out_df["before_top_slope_deg"].abs() + out_df["before_bottom_slope_deg"].abs()
            - out_df["after_top_slope_deg"].abs() - out_df["after_bottom_slope_deg"].abs()
        )

    # 저장
    csv_out = os.path.join(OUT_ROOT, "affine_qc_results.csv")
    xlsx_out = os.path.join(OUT_ROOT, "affine_qc_results.xlsx")
    summary_out = os.path.join(OUT_ROOT, "affine_qc_summary.csv")

    out_df.to_csv(csv_out, index=False, encoding="utf-8-sig")
    out_df.to_excel(xlsx_out, index=False)

    # 요약표
    summary = {
        "n_total": [len(out_df)],
        "n_ok": [int((out_df["status"] == "ok").sum()) if "status" in out_df.columns else 0],
        "n_qc_ok": [int((out_df["qc_flag"] == "ok").sum()) if "qc_flag" in out_df.columns else 0],
        "mean_rot_deg": [float(out_df["rot_deg"].mean()) if "rot_deg" in out_df.columns else np.nan],
        "mean_slope_total_improve": [float(out_df["slope_total_improve"].mean()) if "slope_total_improve" in out_df.columns else np.nan],
    }
    pd.DataFrame(summary).to_csv(summary_out, index=False, encoding="utf-8-sig")

    print("완료")
    print("CSV:", csv_out)
    print("XLSX:", xlsx_out)
    print("SUMMARY:", summary_out)
    print("COMPARE DIR:", compare_dir)
    print("ALIGNED DIR:", aligned_dir)


if __name__ == "__main__":
    main()


affine qc:   0%|          | 0/30 [00:00<?, ?it/s]

완료
CSV: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine_v1/affine_qc_results.csv
XLSX: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine_v1/affine_qc_results.xlsx
SUMMARY: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine_v1/affine_qc_summary.csv
COMPARE DIR: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine_v1/compare
ALIGNED DIR: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine_v1/aligned


실패이후, 회전 기준도 잘못잡았지만 회전이 중요한게 아니라 각 베드 사이의 촬영각이 다를테니 이를 기준으로 정규화해야 한다는걸 깨닫고 2번째에서는 지표 추출 및 정규화 기준을 다시 뽑는거로 바꾸었다



*   1캔버스: geometry table + ref 함수 생성 ✅
*   2캔버스: 지표 정규화 적용 ✅
*   3캔버스: 이미지 약한 보정 실험 ✅

이 구조 딱 맞음.



#2번째: 정규화 기준이 될 지표 뽑기

In [1]:
# -*- coding: utf-8 -*-
"""
[STEP 1-2] Geometry table + reference functions 생성
- 입력: per_image_results_v3.csv
- 필터: matched_count==4 (gold subset)
- 산출:
  1) geometry_table.csv (각 행의 theta_obs, W, H 등)
  2) ref_table.csv (x에 따른 theta_ref, W_ref, H_ref)

요구사항 반영
- 대용량 대비: chunk 처리 + 진행률(ETA)
- 병렬: 계산 가벼워서 불필요 → 대신 I/O 안정성 위주

사용 방법
1) CSV_PATH 수정
2) 실행
3) out_dir에 결과 생성
"""

import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

# ======================
# 경로 설정
# ======================
CSV_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_final/per_image_results_v3.csv"
OUT_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine/v2_geometry"
os.makedirs(OUT_DIR, exist_ok=True)

# ======================
# 설정
# ======================
CHUNK_SIZE = 2000
USE_ONLY_MATCHED4 = True

# ======================
# 1. geometry 계산 함수
# ======================
def compute_geometry(df):
    # width / height
    W = (df["quad_width_top"] + df["quad_width_bottom"]) / 2.0
    H = (df["quad_height_left"] + df["quad_height_right"]) / 2.0

    # 회전각 (상단/하단 평균)
    theta_obs = (df["top_slope_deg"] + df["bottom_slope_deg"]) / 2.0

    out = pd.DataFrame({
        "image_name": df["image_name"],
        "centroid_x": df["centroid_x"],
        "centroid_y": df["centroid_y"],
        "W": W,
        "H": H,
        "theta_obs": theta_obs,
        "width_ratio": df["width_ratio"],
        "height_ratio": df["height_ratio"] if "height_ratio" in df.columns else np.nan
    })

    return out

# ======================
# 2. LOWESS 근사 (간단 구현)
# ======================
def simple_bin_smooth(x, y, n_bins=20):
    """
    LOWESS 대신 간단한 bin 평균
    x를 구간으로 나눠서 평균값 생성
    """
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    df = df.sort_values("x")

    bins = np.linspace(df["x"].min(), df["x"].max(), n_bins+1)
    df["bin"] = np.digitize(df["x"], bins)

    ref = df.groupby("bin").agg({
        "x": "mean",
        "y": "mean"
    }).dropna()

    return ref["x"].values, ref["y"].values

# ======================
# 3. 메인 처리
# ======================
def main():
    print("[START] geometry table 생성")

    reader = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)

    geom_list = []
    total_rows = 0
    t0 = time.time()

    for chunk in tqdm(reader, desc="Processing chunks"):
        if USE_ONLY_MATCHED4:
            chunk = chunk[chunk["matched_count"] == 4]

        geom = compute_geometry(chunk)
        geom_list.append(geom)
        total_rows += len(chunk)

    geom_df = pd.concat(geom_list, ignore_index=True)

    print(f"[INFO] total rows used: {len(geom_df)}")

    # 저장
    geom_path = os.path.join(OUT_DIR, "geometry_table.csv")
    geom_df.to_csv(geom_path, index=False, encoding="utf-8-sig")

    # ======================
    # 4. ref 함수 생성
    # ======================
    print("[START] reference table 생성")

    x = geom_df["centroid_x"].values

    x_theta, theta_ref = simple_bin_smooth(x, geom_df["theta_obs"].values)
    x_w, w_ref = simple_bin_smooth(x, geom_df["W"].values)
    x_h, h_ref = simple_bin_smooth(x, geom_df["H"].values)

    ref_df = pd.DataFrame({
        "x": x_theta,
        "theta_ref": theta_ref,
        "W_ref": np.interp(x_theta, x_w, w_ref),
        "H_ref": np.interp(x_theta, x_h, h_ref)
    })

    ref_path = os.path.join(OUT_DIR, "ref_table.csv")
    ref_df.to_csv(ref_path, index=False, encoding="utf-8-sig")

    print("\n[완료]")
    print("geometry_table:", geom_path)
    print("ref_table:", ref_path)

    print(f"elapsed: {time.time()-t0:.1f}s")


if __name__ == "__main__":
    main()


[START] geometry table 생성


Processing chunks: 1it [00:00, 22.15it/s]

[INFO] total rows used: 728
[START] reference table 생성

[완료]
geometry_table: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine/v2_geometry/geometry_table.csv
ref_table: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine/v2_geometry/ref_table.csv
elapsed: 0.1s


#2.5번째: 정규화 하기

In [2]:
# -*- coding: utf-8 -*-
"""
[STEP 2] 지표 정규화 적용

입력:
- geometry_table.csv
- ref_table.csv

출력:
- normalized_table.csv

핵심:
- 각 행의 centroid_x 기준으로 ref_table에서 보간
- length / area / position 정규화 수행

주의:
- 곱하지 말고 '나눈다'
- 끝단은 과보정 방지 위해 clamp 적용
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# ======================
# 경로
# ======================
GEOM_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine/v2_geometry/geometry_table.csv"
REF_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine/v2_geometry/ref_table.csv"
OUT_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine/v2_geometry/normalized_table.csv"

# ======================
# 설정
# ======================
CLIP_THETA = 3.0   # 회전 최대 보정 각도
EPS = 1e-6

# ======================
# ref 보간 함수
# ======================
def interpolate_ref(x, ref_df):
    theta_ref = np.interp(x, ref_df["x"], ref_df["theta_ref"])
    W_ref = np.interp(x, ref_df["x"], ref_df["W_ref"])
    H_ref = np.interp(x, ref_df["x"], ref_df["H_ref"])
    return theta_ref, W_ref, H_ref

# ======================
# 메인
# ======================
def main():
    print("[START] normalization")

    geom_df = pd.read_csv(GEOM_PATH)
    ref_df = pd.read_csv(REF_PATH)

    results = []

    for _, row in tqdm(geom_df.iterrows(), total=len(geom_df)):
        x = row["centroid_x"]

        theta_ref, W_ref, H_ref = interpolate_ref(x, ref_df)

        # ======================
        # 회전 보정량
        # ======================
        theta_obs = row["theta_obs"]
        delta_theta = theta_ref - theta_obs
        delta_theta = np.clip(delta_theta, -CLIP_THETA, CLIP_THETA)

        # ======================
        # 길이 정규화
        # ======================
        W = row["W"]
        H = row["H"]

        W_norm = W / (W_ref + EPS)
        H_norm = H / (H_ref + EPS)

        # ======================
        # 면적 정규화
        # ======================
        area_raw = W * H
        area_norm = area_raw / (W_ref * H_ref + EPS)

        # ======================
        # 결과 저장
        # ======================
        results.append({
            "image_name": row["image_name"],
            "centroid_x": x,
            "theta_obs": theta_obs,
            "theta_ref": theta_ref,
            "delta_theta": delta_theta,
            "W": W,
            "H": H,
            "W_ref": W_ref,
            "H_ref": H_ref,
            "W_norm": W_norm,
            "H_norm": H_norm,
            "area_norm": area_norm
        })

    out_df = pd.DataFrame(results)
    out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

    print("\n[완료]")
    print("normalized_table:", OUT_PATH)


if __name__ == "__main__":
    main()


[START] normalization


100%|██████████| 728/728 [00:00<00:00, 6585.22it/s]


[완료]
normalized_table: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/front_affine/v2_geometry/normalized_table.csv


#3. 이미지에 대해서 적용하기

In [4]:
# -*- coding: utf-8 -*-
"""
[STEP 3] 이미지 약한 보정 (weak normalization)

입력:
- per_image_results_v3.csv
- ref_table.csv

출력:
- weak_normalized_images/

핵심:
- delta_theta 기반 '약한 회전'
- 선택적으로 '아주 약한 scale'
- 절대 강한 warp/perspective 사용 X

목표:
- geometry residual 제거 (미세 안정화)
- 상추 형태 보존
"""

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# ======================
# 경로
# ======================
CSV_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_final/per_image_results_v3.csv"
REF_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/2. front_affine/v2_geometry/ref_table.csv"
IMG_ROOT = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2"
OUT_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/3. weak_normalized"
os.makedirs(OUT_DIR, exist_ok=True)

# ======================
# 설정
# ======================
CLIP_THETA = 3.0
USE_SCALE = True
SCALE_CLIP = 0.05  # ±5%

# ======================
# ref 보간
# ======================
def interpolate_ref(x, ref_df):
    theta_ref = np.interp(x, ref_df["x"], ref_df["theta_ref"])
    W_ref = np.interp(x, ref_df["x"], ref_df["W_ref"])
    H_ref = np.interp(x, ref_df["x"], ref_df["H_ref"])
    return theta_ref, W_ref, H_ref

# ======================
# 이미지 처리
# ======================
def process_row(row, ref_df):
    img_path = row["image_path"]
    if not os.path.isabs(img_path):
        img_path = os.path.join(IMG_ROOT, img_path)

    if not os.path.exists(img_path):
        return None

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # ======================
    # 회전 계산
    # ======================
    x = row["centroid_x"]
    theta_obs = (row["top_slope_deg"] + row["bottom_slope_deg"]) / 2.0

    theta_ref, W_ref, H_ref = interpolate_ref(x, ref_df)
    delta_theta = theta_ref - theta_obs
    delta_theta = np.clip(delta_theta, -CLIP_THETA, CLIP_THETA)

    # ======================
    # scale 계산
    # ======================
    if USE_SCALE:
        W = (row["quad_width_top"] + row["quad_width_bottom"]) / 2.0
        H = (row["quad_height_left"] + row["quad_height_right"]) / 2.0

        sx = W_ref / (W + 1e-6)
        sy = H_ref / (H + 1e-6)

        sx = np.clip(sx, 1-SCALE_CLIP, 1+SCALE_CLIP)
        sy = np.clip(sy, 1-SCALE_CLIP, 1+SCALE_CLIP)
    else:
        sx, sy = 1.0, 1.0

    # ======================
    # 변환 적용
    # ======================
    center = (w/2, h/2)

    # rotation
    M_rot = cv2.getRotationMatrix2D(center, delta_theta, 1.0)
    img_rot = cv2.warpAffine(img, M_rot, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

    # scale (affine)
    if USE_SCALE:
        M_scale = np.array([[sx, 0, (1-sx)*center[0]],
                            [0, sy, (1-sy)*center[1]]], dtype=np.float32)
        img_out = cv2.warpAffine(img_rot, M_scale, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    else:
        img_out = img_rot

    return img_out

# ======================
# 메인
# ======================
def main():
    df = pd.read_csv(CSV_PATH)
    ref_df = pd.read_csv(REF_PATH)

    # 안정성: matched_count=4만
    df = df[df["matched_count"] == 4]

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_out = process_row(row, ref_df)
        if img_out is None:
            continue

        save_path = os.path.join(OUT_DIR, row["image_name"])
        cv2.imwrite(save_path, img_out)

    print("\n[완료] weak normalization images 생성 완료")


if __name__ == "__main__":
    main()


100%|██████████| 728/728 [06:56<00:00,  1.75it/s]


[완료] weak normalization images 생성 완료
